# 15 — M2 Gate

**Autoridad formal de cierre de M2.** Consume evidencia persistida por 10–14 y decide computablemente si la capa Silver puede congelarse antes de comenzar Gold.

In [1]:
from argos.common.paths import ProjectPaths
from argos.config.silver import load_silver_spec

paths = ProjectPaths.discover(); paths.ensure_runtime_dirs()
SILVER_SPEC = load_silver_spec(paths.configs / "silver.toml")
{"schema_version": SILVER_SPEC.schema_version, "reports": str(paths.reports / "silver")}

{'schema_version': 'm2-silver-v1',
 'reports': '/Users/macm1pro/Desktop/projects/argos/argos-code/reports/silver'}

In [2]:
from argos.evaluation.m2_gate import evaluate_m2_gate, persist_m2_gate

report, decision = evaluate_m2_gate(
    reports_root=paths.reports,
    silver_dir=paths.silver,
    manifest_dir=paths.root / "artifacts" / "manifests",
    schema_version=SILVER_SPEC.schema_version,
)
report

,check_id,category,blocking,passed,observed,requirement
0,notebook_10_blocking_checks,component,True,True,"{""blocking"": 10, ""failed"": [], ""passed"": 10}",notebook 10 has all blocking checks passed
1,notebook_11_blocking_checks,component,True,True,"{""blocking"": 6, ""failed"": [], ""passed"": 6}",notebook 11 has all blocking checks passed
2,notebook_12_blocking_checks,component,True,True,"{""blocking"": 5, ""failed"": [], ""passed"": 5}",notebook 12 has all blocking checks passed
3,notebook_13_blocking_checks,component,True,True,"{""blocking"": 4, ""failed"": [], ""passed"": 4}",notebook 13 has all blocking checks passed
4,notebook_14_cross_validation,validation,True,True,"{""blocking"": 8, ""failed"": [], ""passed"": 8}",notebook 14 cross-Silver validation has no blo...
5,canonical_artifacts_complete,artifacts,True,True,"{""expected"": 12, ""missing"": []}",all M2 canonical Silver artifacts exist
6,lineage_manifests_complete,lineage,True,True,"{""missing"": []}",component and M2 validation manifests are pers...
7,manifest_schema_version_consistent,lineage,True,True,"{""errors"": [], ""expected"": ""m2-silver-v1""}",all M2 manifests declare the active Silver sch...
8,no_blocking_silver_issues,quality,True,True,"{""blocking_issues"": 0, ""warning_issues"": 0}",M2 issue register exists and contains zero blo...
9,entity_resolution_audited,entity_resolution,True,True,"{""rows"": 3, ""statuses"": [""ambiguous_nit"", ""res...",bidder→supplier resolution has an explicit str...


In [3]:
report_path, decision_path = persist_m2_gate(
    report=report,
    decision=decision,
    reports_root=paths.reports,
    decision_path=paths.docs / "decisions" / "M2_GATE.md",
)

print("M2 ✅ CLOSED" if decision.closed else "M2 ❌ OPEN")
print(f"Blocking checks: {decision.blocking_passed}/{decision.blocking_total}")
if decision.failed_checks:
    print("Failed:", ", ".join(decision.failed_checks))
print("Gate report:", report_path)
print("Decision artifact:", decision_path)

assert decision.closed, (
    "M2 remains OPEN. Resolve the failed checks above and rerun the affected notebooks before starting Gold."
)

M2 ✅ CLOSED
Blocking checks: 10/10
Gate report: /Users/macm1pro/Desktop/projects/argos/argos-code/reports/silver/m2_gate_report.parquet
Decision artifact: /Users/macm1pro/Desktop/projects/argos/argos-code/docs/decisions/M2_GATE.md


## Después del cierre

Si el gate imprime `M2 ✅ CLOSED`, congelar `docs/decisions/M2_GATE.md`, los manifests y los reportes Silver. El siguiente milestone es Gold, empezando por target engineering y diseño del split temporal. **No entrenar modelos directamente sobre Silver.**